# Init

This is an example notebook that tries to show how to generate the gds of TWPAs with overlap junctions.
It uses the classes TWPA_elements, TWPA_assembly, TWPA_parameters and CAD_manager.

Up to now (29/11/2024), four different devices have been implemented:
1) Left-Handed TWPAs
2) Right-Handed TWPAs with parallel plate capacitors (with or without modulation)
3) Right-Handed TWPAs with top ground (with or without modulation)
4) SNAIL TWPAs with top ground (with or without modulation)
5) Lumped Elemenet Resonators

The devices and fabrication parameters are defined in the json files fab_parameters and device_parameters. 
These parameters are passed to the class TWPA_parameters that returns design parameters (like dimensions of capacitors) depending on the input ones.

All the design parameters are then passed to the TWPA_elements class the generate the desired unit cell. This is used by the TWPA_assembly class to generate the device.

The generation of the gds and of the job and batch files passes through the CAD_manager class. 

In [1]:
import gdstk
import numpy as np
import os
import importlib
from pprint import pprint
import json

import sys

In [2]:
import TWPA_elements as TWPA_elements_cls
import CAD_manager as CAD_manager_cls
import TWPA_parameters as TWPA_parameters_cls

In [3]:
date = 'yymmdd'
WaferName = 'example'
WaferNumber = '1'

In [4]:
importlib.reload(CAD_manager_cls)
CAD_manager = CAD_manager_cls.CAD_manager(date, WaferName, WaferNumber)

Junk folder already exists, cleaning it...
Junk folder already exists, cleaning it...


# Markers

In [8]:
CAD_manager.generate_markers()

# Litho and doses definition

Let's define the layers where we are going to put the structures on

In [5]:
ground_layer = 1
pad_bottom_layer = 2
connection_wire_bottom_layer = 3
jj_bottom_layer = 4
capa_bottom_layer = 5
jj_top_layer = 6
capa_top_layer = 7
pad_top_layer = 8
connection_wire_top_layer = 9
small_jj_top_layer = 10

And the doses we are going to use to write each layer

In [6]:
dose_matric = {str(ground_layer):3,
               str(pad_bottom_layer):15,
               str(connection_wire_bottom_layer):14,
               str(jj_bottom_layer):14,
               str(capa_bottom_layer):12,
               str(jj_top_layer):14,
               str(capa_top_layer):12,
               str(pad_top_layer):15,
               str(connection_wire_top_layer):14,
               str(small_jj_top_layer):18,
              }

Now we can prepare the lithography plan and associate each layer to a job

In [7]:
litho_plan = {'ground':[ground_layer],
              'pads_bottom_layer':[pad_bottom_layer,connection_wire_bottom_layer],
              'bottom_layer':[jj_bottom_layer,capa_bottom_layer],
              'junctions_top_layer':[jj_top_layer, small_jj_top_layer],
              'capacitors_top_layer':[capa_top_layer],
              'pads_top_layer':[pad_top_layer,connection_wire_top_layer],
             }

And now we can define the batch plan, we have to associate to each batch the corresponding jobs and decide the currents used to write each job.
It's also possible to specify the datum and the sleep time in minutes between the jobs

In [8]:
batch_plan = {'ground':{'jobs':['ground'],                                                                                                               
                        'current':[15],
                        'datum':[8],
                        'sleep':[10]},
              'bottom_layer':{'jobs':['bottom_layer','pad_bottom_layer'],
                              'current':[5,15],
                              'datum':[8,8],
                              'sleep':[10,20]},
              'junctions_top_layer':{'jobs':['junctions_top_layer', 'small_jj_top_layer'],
                              'current':[5,15],
                              'datum':[8,8],
                              'sleep':[10,20]},
              'capa_top_layer':{'jobs':['pad_top_layer','capa_top_layer'],
                          'current':[5,15],
                          'datum':[8,8],
                          'sleep':[10,20]},
             }

# Design generation

Now we can create the design, we can both add different devices chip by chip or use for loop to design a all wafer with the same devices.
Let's go chip by chip:
1) We will add a LH TWPA on chip 00
2) Then a RH TWPA on chip 01 and 02
3) And LERs on chip 03

## Left-Handed Josephson Transmission Line

In [10]:
importlib.reload(TWPA_parameters_cls)
CAD_manager.load_reset_libs()

In [11]:
# Type of device
CAD_manager.device_type = 'LH'
device_version = '01'

In [12]:
# Chip
row = 0
column = 0

In [13]:
### Fab parameters file
dirname = os.path.abspath('')
fab_param_filename = os.path.join(dirname, 'default_parameters\\fab_parameters.json')
with open(fab_param_filename) as json_file:
    fab_parameters = json.load(json_file)


In [14]:
### Device parameters file
dirname = os.path.abspath('')
device_param_filename = os.path.join(dirname, 'default_parameters\\device_parameters.json')
with open(device_param_filename) as json_file:
    device_parameter = json.load(json_file)

In [15]:
# CAD manager general info
CAD_manager.modulation = False
CAD_manager.litho_plan = litho_plan
CAD_manager.dose_matric = dose_matric
CAD_manager.chip = str(row)+str(column)

In [16]:
# Device parameters
params = TWPA_parameters_cls.LH_parameters(fab_parameters, device_parameter[CAD_manager.device_type])
print('\n'+'\033[1m'+'Chip '+CAD_manager.chip+'\033[0m')
number_of_junctions, area_capa, f0 = params.get_params(print_params=True)


Chip 00
 Parameters 
Junction capacitance per unit area: c_j = 45.00 fF/um^2
Junction critical current density: j_c = 50.00 A/cm^2
Capacitor capacitance per unit area: c_c = 8.63 fF/um^2
Number of cells: Ncell = 600
Number of junctions to ground: Nj_ground = 14
Number of capacitors per cell: Ncapa = 2
Junction dimensions: H = 7.00 um | W = 1.00 um | A = 7.00 um^2
Room temp resistance: Rj = 72.50 Ohm
Critical current: Ic = 2.69 uA
Normal state resistance: Rn = 122.52 Ohm
Josephson inductance: Lj = 122.24 pH
Josephson capacitance: Cj = 315.00 fF
Inductance to ground: Lj_ground = 1.71 nH
Capacitance to ground: Cj_ground = 22.50 fF
Series capacitance: C = 684.54 fF
Capacitor area: A = 158.64 um^2
Plasma frequency: fj = 25.65 GHz
Cut-off frequency: f0 = 4.65 GHz


In [17]:
# Element design definition
LH = TWPA_elements_cls.LHcell()
LH.litho_overlap = 0.1
LH.etching_offset = 0.5
LH.electrode_height_difference_capa = 5
LH.electrode_height_difference_JJ = 0.5
LH.y_low_current_ground = 3
LH.number_of_capacitors = int(device_parameter[CAD_manager.device_type]['capacitor_parameters']['number_of_capacitors_per_unit_cell'])
LH.width_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction'] 
LH.height_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction']
LH.spacing_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['spacing_between_junctions']
LH.number_of_junctions = number_of_junctions
LH.width_capa = device_parameter[CAD_manager.device_type]['capacitor_parameters']['width_of_capacitor']
LH.height_capa = area_capa/LH.width_capa
LH.new_area_capa = area_capa
LH.spacing_capa = device_parameter[CAD_manager.device_type]['capacitor_parameters']['spacing_between_capa']    

In [18]:
#  Layer definition
LH.ground_layer = ground_layer
CAD_manager.pad_bottom_layer = pad_bottom_layer
CAD_manager.connection_wire_bottom_layer = connection_wire_bottom_layer
LH.jj_bottom_layer = jj_bottom_layer
LH.capa_bottom_layer = capa_bottom_layer
LH.jj_top_layer = jj_top_layer
LH.capa_top_layer = capa_top_layer
CAD_manager.pad_top_layer = pad_top_layer
CAD_manager.connection_wire_top_layer = connection_wire_top_layer
CAD_manager.ground_layer = LH.ground_layer

In [19]:
# CAD manager device info
CAD_manager.n_unit_cells = device_parameter[CAD_manager.device_type]['TWPA_parameters']['number_of_cells']
CAD_manager.element = LH
CAD_manager.dev_label = CAD_manager.device_type + '_V' + device_version + '_' + CAD_manager.chip
CAD_manager.dev_label += ' | ' + str(CAD_manager.n_unit_cells) + ' | ' + str(f0)
CAD_manager.design_filename = CAD_manager.device_type + '_' + CAD_manager.chip

In [20]:
# gds and job file generation
CAD_manager.generate_GDS()
CAD_manager.populate_dose_matric()
CAD_manager.generate_jobs(row=row, column=column)

## Right-Handed Josephson Transmission Line

### With parallel plate capacitors

#### No modulation

In [9]:
importlib.reload(TWPA_parameters_cls)
CAD_manager.load_reset_libs()

In [10]:
# Type of device
CAD_manager.device_type = 'RH'
device_version = '01'

In [11]:
# Chip
row = 0
column = 1

In [12]:
### Fab parameters file
dirname = os.path.abspath('')
fab_param_filename = os.path.join(dirname, 'default_parameters\\fab_parameters.json')
with open(fab_param_filename) as json_file:
    fab_parameters = json.load(json_file)

In [13]:
### Device parameters file
dirname = os.path.abspath('')
device_param_filename = os.path.join(dirname, 'default_parameters\\device_parameters.json')
with open(device_param_filename) as json_file:
    device_parameter = json.load(json_file)

In [14]:
# CAD manager general info
CAD_manager.modulation = False
CAD_manager.litho_plan = litho_plan
CAD_manager.dose_matric = dose_matric
CAD_manager.chip = str(row)+str(column)

In [15]:
# Device parameters
params = TWPA_parameters_cls.RH_parameters(fab_parameters, device_parameter[CAD_manager.device_type])
print('\n'+'\033[1m'+'Chip '+CAD_manager.chip+'\033[0m')
area_capa = params.get_params(print_params=True)


Chip 01
Junction capacitance per unit area: c_j = 45.00 fF/um^2
Junction critical current density: j_c = 50.00 A/cm^2
Capacitor capacitance per unit area: c_c = 8.63 fF/um^2
Number of cells: Ncell = 900
Number of junctions per cell: Njj = 2
Number of capacitors per cell: Ncapa = 2
Junction dimensions: H = 3.50 um | W = 2.00 um | A = 7.00 um^2
Room temp resistance: Rj = 72.50 Ohm
Critical current: Ic = 2.69 uA
Normal state resistance: Rn = 122.52 Ohm
Josephson inductance: Lj = 122.24 pH
Josephson capacitance: Cj = 315.00 fF
Josephson inductance per unit cell: Lj_cell = 244.48 pH
Josephson capacitance per unit cell: Cj_cell = 157.50 fF
Ground capacitance to get 50 Ohm matching: Cg = 97.79 fF
Capacitor area: A = 11.33 um^2
Plasma frequency: fj = 25.65 GHz
Cut-off frequency: f0 = 32.55 GHz
No modulation


In [16]:
# Element design definition
RH = TWPA_elements_cls.RHcell()
RH.litho_overlap = 0.1
RH.etching_offset = 0.25
RH.electrode_height_difference_jj = 0.5
RH.y_low_current_ground = 3
RH.width_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction'] 
RH.height_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction']
RH.spacing_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['spacing_between_junctions'] 
RH.number_of_junctions = device_parameter[CAD_manager.device_type]['junction_parameters']['number_of_junctions_per_unit_cell'] 
RH.area_capa = area_capa
RH.spacing_capa = device_parameter[CAD_manager.device_type]['capacitor_parameters']['spacing_between_capacitors'] 
RH.electrode_height_difference_capa = 1
RH.number_of_capacitors = device_parameter[CAD_manager.device_type]['capacitor_parameters']['number_of_capacitors_per_unit_cell'] 


In [17]:
# Layers definition
RH.ground_layer = ground_layer
CAD_manager.pad_bottom_layer = pad_bottom_layer
CAD_manager.connection_wire_bottom_layer = connection_wire_bottom_layer
RH.jj_bottom_layer = jj_bottom_layer
RH.capa_bottom_layer = capa_bottom_layer
RH.jj_top_layer = jj_top_layer
RH.capa_top_layer = capa_top_layer
CAD_manager.pad_top_layer = pad_top_layer
CAD_manager.connection_wire_top_layer = connection_wire_top_layer
CAD_manager.ground_layer = RH.ground_layer

In [18]:
# CAD manager device info
CAD_manager.n_unit_cells = device_parameter[CAD_manager.device_type]['TWPA_parameters']['number_of_cells']
CAD_manager.element = RH
CAD_manager.dev_label = CAD_manager.device_type + '_V' + device_version + '_' + CAD_manager.chip
CAD_manager.dev_label += ' | ' + str(CAD_manager.n_unit_cells)
CAD_manager.design_filename = CAD_manager.device_type + '_' + CAD_manager.chip

In [19]:
# gds and job files generation
CAD_manager.generate_GDS()
CAD_manager.populate_dose_matric()
CAD_manager.generate_jobs(row=row, column=column)

##### Change area capacitors

Let's imagine we already deposited our junctions and did the DC test from which we extracted a different crittical current density respect to the one reported in the fab_parameter json file.
It's possible to change the area of the capacitors without changing anything else in the design to still have 50 Ohm matching. To do so we need to update the value of the jc, pass the previous value of the capacitor area to the variable element.area_capa and set the change_area_capa variable to True.

In [32]:
importlib.reload(TWPA_parameters_cls)
CAD_manager.load_reset_libs()

In [33]:
# New parameters
change_area_capa = True
new_jc = 39 #A/cm^2

# Old parameters
old_area_capa = 10.9

In [34]:
# Type of device
CAD_manager.device_type = 'RH'
device_version = '01'

In [35]:
# Chip
row = 0
column = 1

In [36]:
### Fab parameters file
dirname = os.path.abspath('')
fab_param_filename = os.path.join(dirname, 'default_parameters\\fab_parameters.json')
with open(fab_param_filename) as json_file:
    fab_parameters = json.load(json_file)

In [37]:
### Device parameters file
dirname = os.path.abspath('')
device_param_filename = os.path.join(dirname, 'default_parameters\\device_parameters.json')
with open(device_param_filename) as json_file:
    device_parameter = json.load(json_file)

In [38]:
# Update parameter
fab_parameters['junction_parameters']['critical_current_density'] = new_jc

In [39]:
# CAD manager general info
CAD_manager.modulation = False
CAD_manager.litho_plan = litho_plan
CAD_manager.dose_matric = dose_matric
CAD_manager.chip = str(row)+str(column)

In [40]:
# Device parameters
params = TWPA_parameters_cls.RH_parameters(fab_parameters, device_parameter[CAD_manager.device_type])
print('\n'+'\033[1m'+'Chip '+CAD_manager.chip+'\033[0m')
area_capa = params.get_params(print_params=True)


Chip 01
Junction capacitance per unit area: c_j = 45.00 fF/um^2
Junction critical current density: j_c = 39.00 A/cm^2
Capacitor capacitance per unit area: c_c = 8.63 fF/um^2
Number of cells: Ncell = 900
Number of junctions per cell: Njj = 2
Number of capacitors per cell: Ncapa = 2
Junction dimensions: H = 3.50 um | W = 2.00 um | A = 7.00 um^2
Room temp resistance: Rj = 92.95 Ohm
Critical current: Ic = 2.10 uA
Normal state resistance: Rn = 157.08 Ohm
Josephson inductance: Lj = 156.72 pH
Josephson capacitance: Cj = 315.00 fF
Josephson inductance per unit cell: Lj_cell = 313.43 pH
Josephson capacitance per unit cell: Cj_cell = 157.50 fF
Ground capacitance to get 50 Ohm matching: Cg = 125.37 fF
Capacitor area: A = 14.53 um^2
Plasma frequency: fj = 22.65 GHz
Cut-off frequency: f0 = 25.39 GHz
No modulation


In [41]:
# Element design definition
RH = TWPA_elements_cls.RHcell()

RH.litho_overlap = 0.1
RH.etching_offset = 0.25
RH.electrode_height_difference_jj = 0.5
RH.y_low_current_ground = 3
RH.width_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction'] 
RH.height_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction']
RH.spacing_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['spacing_between_junctions'] 
RH.number_of_junctions = device_parameter[CAD_manager.device_type]['junction_parameters']['number_of_junctions_per_unit_cell'] 
RH.spacing_capa = device_parameter[CAD_manager.device_type]['capacitor_parameters']['spacing_between_capacitors'] 
RH.electrode_height_difference_capa = 1
RH.number_of_capacitors = device_parameter[CAD_manager.device_type]['capacitor_parameters']['number_of_capacitors_per_unit_cell'] 
RH.area_capa = old_area_capa
RH.new_area_capa = area_capa

In [42]:
# Layer definition
RH.ground_layer = ground_layer
CAD_manager.pad_bottom_layer = pad_bottom_layer
CAD_manager.connection_wire_bottom_layer = connection_wire_bottom_layer
RH.jj_bottom_layer = jj_bottom_layer
RH.capa_bottom_layer = capa_bottom_layer
RH.jj_top_layer = jj_top_layer
RH.capa_top_layer = capa_top_layer
CAD_manager.pad_top_layer = pad_top_layer
CAD_manager.connection_wire_top_layer = connection_wire_top_layer
CAD_manager.ground_layer = RH.ground_layer

In [43]:
# CAD manager device info
CAD_manager.n_unit_cells = device_parameter[CAD_manager.device_type]['TWPA_parameters']['number_of_cells']
CAD_manager.element = RH
CAD_manager.dev_label = CAD_manager.device_type + '_V' + device_version + '_' + CAD_manager.chip
CAD_manager.dev_label += ' | ' + str(CAD_manager.n_unit_cells)
CAD_manager.design_filename = CAD_manager.device_type + '_' + CAD_manager.chip

In [44]:
# gds and job file generation
CAD_manager.generate_GDS(change_area_capa=change_area_capa)
CAD_manager.populate_dose_matric()
CAD_manager.generate_jobs(row=row, column=column)

#### Impedance modulation

In [45]:
importlib.reload(TWPA_parameters_cls)
CAD_manager.load_reset_libs()

In [46]:
TWPA_modulation = True

In [47]:
# Type of device
CAD_manager.device_type = 'RH'
device_version = '01'

In [48]:
# Chip
row = 0
column = 2

In [49]:
### Fab parameters file
dirname = os.path.abspath('')
fab_param_filename = os.path.join(dirname, 'default_parameters\\fab_parameters.json')
with open(fab_param_filename) as json_file:
    fab_parameters = json.load(json_file)

In [50]:
### Device parameters file
dirname = os.path.abspath('')
device_param_filename = os.path.join(dirname, 'default_parameters\\device_parameters.json')
with open(device_param_filename) as json_file:
    device_parameter = json.load(json_file)

In [51]:
# CAD manager general info
CAD_manager.modulation = TWPA_modulation
CAD_manager.litho_plan = litho_plan
CAD_manager.dose_matric = dose_matric
CAD_manager.chip = str(row)+str(column)

In [52]:
# Device parameters
params = TWPA_parameters_cls.RH_parameters(fab_parameters, device_parameter[CAD_manager.device_type])
print('\n'+'\033[1m'+'Chip '+CAD_manager.chip+'\033[0m')
area_capa = params.get_params(modulation=CAD_manager.modulation, print_params=True)


Chip 02
Junction capacitance per unit area: c_j = 45.00 fF/um^2
Junction critical current density: j_c = 50.00 A/cm^2
Capacitor capacitance per unit area: c_c = 8.63 fF/um^2
Number of cells: Ncell = 900
Number of junctions per cell: Njj = 2
Number of capacitors per cell: Ncapa = 2
Junction dimensions: H = 3.50 um | W = 2.00 um | A = 7.00 um^2
Room temp resistance: Rj = 72.50 Ohm
Critical current: Ic = 2.69 uA
Normal state resistance: Rn = 122.52 Ohm
Josephson inductance: Lj = 122.24 pH
Josephson capacitance: Cj = 315.00 fF
Josephson inductance per unit cell: Lj_cell = 244.48 pH
Josephson capacitance per unit cell: Cj_cell = 157.50 fF
Ground capacitance to get 50 Ohm matching: Cg = 97.79 fF
Capacitor area: A = 11.33 um^2
Plasma frequency: fj = 25.65 GHz
Cut-off frequency: f0 = 32.55 GHz
Modulation period: Np = 16
Modulation amplitude: eta = 5%
Gap frequency: fgap = 6.20 GHz


In [53]:
# Element design definition
RH = TWPA_elements_cls.RHcell()

RH.litho_overlap = 0.1
RH.etching_offset = 0.25
RH.electrode_height_difference_jj = 0.5
RH.y_low_current_ground = 3
RH.width_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction'] 
RH.height_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction']
RH.spacing_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['spacing_between_junctions'] 
RH.number_of_junctions = device_parameter[CAD_manager.device_type]['junction_parameters']['number_of_junctions_per_unit_cell'] 
RH.area_capa = area_capa
RH.spacing_capa = device_parameter[CAD_manager.device_type]['capacitor_parameters']['spacing_between_capacitors'] 
RH.electrode_height_difference_capa = 1
RH.number_of_capacitors = device_parameter[CAD_manager.device_type]['capacitor_parameters']['number_of_capacitors_per_unit_cell'] 

In [54]:
# Layer definition
RH.ground_layer = ground_layer
CAD_manager.pad_bottom_layer = pad_bottom_layer
CAD_manager.connection_wire_bottom_layer = connection_wire_bottom_layer
RH.jj_bottom_layer = jj_bottom_layer
RH.capa_bottom_layer = capa_bottom_layer
RH.jj_top_layer = jj_top_layer
RH.capa_top_layer = capa_top_layer
CAD_manager.pad_top_layer = pad_top_layer
CAD_manager.connection_wire_top_layer = connection_wire_top_layer
CAD_manager.ground_layer = RH.ground_layer

In [55]:
# CAD manager device info
CAD_manager.element = RH
CAD_manager.n_unit_cells = device_parameter[CAD_manager.device_type]['TWPA_parameters']['number_of_cells']
CAD_manager.modulation_period = device_parameter[CAD_manager.device_type]['TWPA_parameters']['modulation_period']
CAD_manager.modulation_amplitude_percent = device_parameter[CAD_manager.device_type]['TWPA_parameters']['modulation_amplitude_percent']
CAD_manager.dev_label = CAD_manager.device_type + '_V' + device_version + '_' + CAD_manager.chip
CAD_manager.dev_label += ' | ' + str(CAD_manager.n_unit_cells) + ' | '+str(CAD_manager.modulation_period)+' | '+str(CAD_manager.modulation_amplitude_percent)+'%'
CAD_manager.design_filename = CAD_manager.device_type + '_' + CAD_manager.chip

In [56]:
# gds and job file generation
CAD_manager.generate_GDS()
CAD_manager.populate_dose_matric()
CAD_manager.generate_jobs(row=row, column=column)

### With top ground

If a given device on the wafer has different number of layers and/or doses we can change the dose_matric variable

In [9]:
dose_matric = {str(ground_layer):3,
               str(pad_bottom_layer):15,
               str(connection_wire_bottom_layer):14,
               str(jj_bottom_layer):14,
               str(jj_top_layer):14,
               str(pad_top_layer):15,
               str(connection_wire_top_layer):14,
              }

#### No modulation

In [10]:
importlib.reload(TWPA_parameters_cls)
CAD_manager.load_reset_libs()

In [11]:
# Type of device
CAD_manager.device_type = 'SJ'
device_version = '01'

In [12]:
# Chip
row = 0
column = 3

In [13]:
### Fab parameters file
dirname = os.path.abspath('')
fab_param_filename = os.path.join(dirname, 'default_parameters\\fab_parameters.json')
with open(fab_param_filename) as json_file:
    fab_parameters = json.load(json_file)

In [14]:
### Device parameters file
dirname = os.path.abspath('')
device_param_filename = os.path.join(dirname, 'default_parameters\\device_parameters.json')
with open(device_param_filename) as json_file:
    device_parameter = json.load(json_file)

In [15]:
# CAD manager general info
CAD_manager.modulation = False
CAD_manager.litho_plan = litho_plan
CAD_manager.dose_matric = dose_matric
CAD_manager.chip = str(row)+str(column)

In [16]:
# Device parameters
params = TWPA_parameters_cls.SJ_parameters(fab_parameters, device_parameter[CAD_manager.device_type])
print('\n'+'\033[1m'+'Chip '+CAD_manager.chip+'\033[0m')
Cg = params.get_params(print_params=True)


Chip 03
Junction capacitance per unit area: c_j = 45.00 fF/um^2
Junction critical current density: j_c = 50.00 A/cm^2
Number of cells: Ncell = 2000
Number of junctions per cell: Njj = 1
Junction dimensions: H = 7.00 um | W = 1.00 um | A = 7.00 um^2
Room temp resistance: Rj = 72.50 Ohm
Critical current: Ic = 2.69 uA
Normal state resistance: Rn = 122.52 Ohm
Josephson inductance: Lj = 122.24 pH
Josephson capacitance: Cj = 315.00 fF
Josephson inductance per unit cell: Lj_cell = 122.24 pH
Josephson capacitance per unit cell: Cj_cell = 315.00 fF
Ground capacitance to get 50 Ohm matching: Cg = 48.90 fF
Plasma frequency: fj = 25.65 GHz
Cut-off frequency: f0 = 65.10 GHz
No modulation


In [17]:
# Element design definition
SJ = TWPA_elements_cls.Overlap_jj()
SJ.litho_overlap = 0.1
SJ.etching_offset = 0.25
SJ.electrode_height_difference = 0.5
SJ.width_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction'] 
SJ.height_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction']
SJ.spacing_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['spacing_between_junctions'] 

In [18]:
# Layers definition
CAD_manager.pad_bottom_layer = pad_bottom_layer
CAD_manager.connection_wire_bottom_layer = connection_wire_bottom_layer
SJ.jj_bottom_layer = jj_bottom_layer
SJ.jj_top_layer = jj_top_layer
CAD_manager.pad_top_layer = pad_top_layer
CAD_manager.connection_wire_top_layer = connection_wire_top_layer
CAD_manager.ground_layer = ground_layer

In [19]:
# CAD manager device info
CAD_manager.n_unit_cells = device_parameter[CAD_manager.device_type]['TWPA_parameters']['number_of_cells']
CAD_manager.element = SJ
CAD_manager.dev_label = CAD_manager.device_type + '_V' + device_version + '_' + CAD_manager.chip
CAD_manager.dev_label += ' | ' + str(CAD_manager.n_unit_cells)
CAD_manager.design_filename = CAD_manager.device_type + '_' + CAD_manager.chip

In [20]:
# gds and job files generation
CAD_manager.generate_GDS()
CAD_manager.populate_dose_matric()
CAD_manager.generate_jobs(row=row, column=column)

#### Impedance modulation

In [21]:
importlib.reload(TWPA_parameters_cls)
CAD_manager.load_reset_libs()

In [22]:
TWPA_modulation = True

In [23]:
# Type of device
CAD_manager.device_type = 'SJ'
device_version = '01'

In [24]:
# Chip
row = 1
column = 0

In [25]:
### Fab parameters file
dirname = os.path.abspath('')
fab_param_filename = os.path.join(dirname, 'default_parameters\\fab_parameters.json')
with open(fab_param_filename) as json_file:
    fab_parameters = json.load(json_file)

In [26]:
### Device parameters file
dirname = os.path.abspath('')
device_param_filename = os.path.join(dirname, 'default_parameters\\device_parameters.json')
with open(device_param_filename) as json_file:
    device_parameter = json.load(json_file)

In [27]:
# CAD manager general info
CAD_manager.modulation = TWPA_modulation
CAD_manager.litho_plan = litho_plan
CAD_manager.dose_matric = dose_matric
CAD_manager.chip = str(row)+str(column)

In [28]:
# Device parameters
params = TWPA_parameters_cls.SJ_parameters(fab_parameters, device_parameter[CAD_manager.device_type])
print('\n'+'\033[1m'+'Chip '+CAD_manager.chip+'\033[0m')
Cg = params.get_params(modulation=CAD_manager.modulation, print_params=True)


Chip 10
Junction capacitance per unit area: c_j = 45.00 fF/um^2
Junction critical current density: j_c = 50.00 A/cm^2
Number of cells: Ncell = 2000
Number of junctions per cell: Njj = 1
Junction dimensions: H = 7.00 um | W = 1.00 um | A = 7.00 um^2
Room temp resistance: Rj = 72.50 Ohm
Critical current: Ic = 2.69 uA
Normal state resistance: Rn = 122.52 Ohm
Josephson inductance: Lj = 122.24 pH
Josephson capacitance: Cj = 315.00 fF
Josephson inductance per unit cell: Lj_cell = 122.24 pH
Josephson capacitance per unit cell: Cj_cell = 315.00 fF
Ground capacitance to get 50 Ohm matching: Cg = 48.90 fF
Plasma frequency: fj = 25.65 GHz
Cut-off frequency: f0 = 65.10 GHz
Modulation period: Np = 16
Modulation amplitude: eta = 5%
Gap frequency: fgap = 11.44 GHz


In [29]:
# Element design definition
SJ = TWPA_elements_cls.Overlap_jj()
SJ.litho_overlap = 0.1
SJ.etching_offset = 0.25
SJ.electrode_height_difference = 0.5
SJ.width_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['width_of_junction'] 
SJ.height_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['height_of_junction']
SJ.spacing_jj = device_parameter[CAD_manager.device_type]['junction_parameters']['spacing_between_junctions'] 

In [30]:
# Layer definition
CAD_manager.pad_bottom_layer = pad_bottom_layer
CAD_manager.connection_wire_bottom_layer = connection_wire_bottom_layer
SJ.jj_bottom_layer = jj_bottom_layer
SJ.jj_top_layer = jj_top_layer
CAD_manager.pad_top_layer = pad_top_layer
CAD_manager.connection_wire_top_layer = connection_wire_top_layer
CAD_manager.ground_layer = ground_layer

In [31]:
# CAD manager device info
CAD_manager.element = SJ
CAD_manager.n_unit_cells = device_parameter[CAD_manager.device_type]['TWPA_parameters']['number_of_cells']
CAD_manager.modulation_period = device_parameter[CAD_manager.device_type]['TWPA_parameters']['modulation_period']
CAD_manager.modulation_amplitude_percent = device_parameter[CAD_manager.device_type]['TWPA_parameters']['modulation_amplitude_percent']
CAD_manager.dev_label = CAD_manager.device_type + '_V' + device_version + '_' + CAD_manager.chip
CAD_manager.dev_label += ' | ' + str(CAD_manager.n_unit_cells) + ' | '+str(CAD_manager.modulation_period)+' | '+str(CAD_manager.modulation_amplitude_percent)+'%'
CAD_manager.design_filename = CAD_manager.device_type + '_' + CAD_manager.chip

In [32]:
# gds and job file generation
CAD_manager.generate_GDS()
CAD_manager.populate_dose_matric()
CAD_manager.generate_jobs(row=row, column=column)

## Overlap SNAIL chain with top ground

In [10]:
importlib.reload(TWPA_parameters_cls)
CAD_manager.load_reset_libs()

In [11]:
dose_matric = {str(ground_layer):1,
               str(pad_bottom_layer):15,
               str(connection_wire_bottom_layer):15,
               str(jj_bottom_layer):16,
               str(jj_top_layer):16,
               str(pad_top_layer):15,
               str(connection_wire_top_layer):15,
               str(small_jj_top_layer):20,
              }

### No modulation

In [12]:
# Type of device
CAD_manager.device_type = 'SNAIL'
device_version = '01'

In [84]:
# Chip
row = 1
column = 1

In [85]:
### Fab parameters file
dirname = os.path.abspath('')
fab_param_filename = os.path.join(dirname, 'default_parameters\\fab_parameters.json')
with open(fab_param_filename) as json_file:
    fab_parameters = json.load(json_file)

In [86]:
### Device parameters file
dirname = os.path.abspath('')
device_param_filename = os.path.join(dirname, 'default_parameters\\device_parameters.json')
with open(device_param_filename) as json_file:
    device_parameter = json.load(json_file)

In [87]:
# CAD manager general info
CAD_manager.modulation = False
CAD_manager.litho_plan = litho_plan
CAD_manager.dose_matric = dose_matric
CAD_manager.chip = str(row)+str(column)

In [88]:
device_parameter[CAD_manager.device_type]

{'design_SNAIL_parameters': 'Design parameters for a RH SNAIL TWPA with top ground',
 'large_junction_parameters': {'number_of_SNAILs_per_unit_cell': 2,
  'number_of_large_junctions_in_the_SNAIL': 3,
  'width_of_junction': 1,
  'height_of_junction': 4,
  'spacing_between_junctions': 1},
 'small_junction_parameters': {'width_of_junction': 1,
  'height_of_junction': 0.7,
  'fixed_dimension': 'Height',
  'critical_current_ratio_large_small': 0.1},
 'ground_capacitance_pads_parameters': {'width_of_pads': 0.75,
  'ALD_diel_thickness_nm': 60,
  'flux_impedance_matching': 0.37},
 'loop_parameters': {'loop_area': 24},
 'TWPA_parameters': {'number_of_cells': 700,
  'inverted_cells': 'False',
  'modulate_capa_only': 'False',
  'keep_Z_constant_with_modulation': 'False',
  'modulation_period': 8,
  'modulation_amplitude_percent': 10}}

In [89]:
# Device parameters
importlib.reload(TWPA_parameters_cls)
importlib.reload(TWPA_elements_cls)
params = TWPA_parameters_cls.SNAIL_parameters(fab_parameters, device_parameter[CAD_manager.device_type])
print('\n'+'\033[1m'+'Chip '+CAD_manager.chip+'\033[0m')
Cg, total_ground_area, Hj_small, Wj_small, area_ratio = params.get_params(print_params=True)


Chip 11
Junction capacitance per unit area: c_j = 45.00 fF/um^2
Large junction critical current density: j_c = 90.00 A/cm^2
Small junction critical current density: j_c = 105.00 A/cm^2
Number of cells: Ncell = 700
Number of large junctions in SNAIL: Njj = 3
Number of junctions per cell: Njj = 2
Large Junction dimensions: H = 4.00 um | W = 1.00 um | A = 4.00 um^2
Small Junction dimensions: H = 0.70 um | W = 0.49 um | A = 0.34 um^2
Critical current ratio: r_Ic = 0.10 
Area ratio: r_A = 0.09 
Critical current large: Ic = 2.77 uA
Critical current small: Ic = 0.28 uA
Josephson inductance per unit cell: Lj_cell @ 0.37 $\Phi_0$ = 416.88 pH
Josephson capacitance per unit cell: Cj_cell = 75.43 fF
Ground capacitance to get 50 Ohm matching: Cg = 166.75 fF
Total area to ground: 115.31 um^2
Plasma frequency: fj = 28.38 GHz
Cut-off frequency: f0 = 19.09 GHz
No modulation


In [90]:
# Element design definition
SNAIL = TWPA_elements_cls.SNAILcell()
SNAIL.modulation = 'False'
SNAIL.litho_overlap = 0.1
SNAIL.etching_offset = 0.25
SNAIL.area_ratio = area_ratio
SNAIL.width_jj_large = device_parameter[CAD_manager.device_type]['large_junction_parameters']['width_of_junction']
SNAIL.height_jj_large = device_parameter[CAD_manager.device_type]['large_junction_parameters']['height_of_junction']
SNAIL.spacing_jj_large = device_parameter[CAD_manager.device_type]['large_junction_parameters']['spacing_between_junctions']
SNAIL.electrode_height_difference_jj = 0.5
SNAIL.number_of_large_junctions = device_parameter[CAD_manager.device_type]['large_junction_parameters']['number_of_large_junctions_in_the_SNAIL']

SNAIL.width_jj_small = Wj_small
SNAIL.height_jj_small = Hj_small

SNAIL.loop_area = device_parameter[CAD_manager.device_type]['loop_parameters']['loop_area']
SNAIL.loop_arm_width = SNAIL.spacing_jj_large/2
SNAIL.loop_width = SNAIL.number_of_large_junctions * (SNAIL.width_jj_large + SNAIL.spacing_jj_large) # Defined as is because the size of bottom loop arms is calculated upon the size of a JJ element (1*JJ_width + 2*JJ_spacing)
SNAIL.loop_height = SNAIL.loop_area / (SNAIL.loop_width - SNAIL.loop_arm_width)



upper_area = SNAIL.number_of_large_junctions * (SNAIL.width_jj_large + SNAIL.spacing_jj_large/2) * (2*SNAIL.electrode_height_difference_jj + SNAIL.height_jj_large) + SNAIL.number_of_large_junctions * SNAIL.height_jj_large * SNAIL.spacing_jj_large/2

loop_bridges_area = SNAIL.loop_arm_width*SNAIL.loop_height + SNAIL.loop_arm_width*(SNAIL.loop_height + SNAIL.height_jj_large + 2*SNAIL.electrode_height_difference_jj)

lower_area = (SNAIL.loop_width + SNAIL.loop_arm_width)*(SNAIL.height_jj_small + 2*SNAIL.electrode_height_difference_jj) - 2*SNAIL.electrode_height_difference_jj*(SNAIL.spacing_jj_large/2 - SNAIL.litho_overlap/2)


pads_area = total_ground_area - (upper_area + loop_bridges_area + lower_area)
pads_height = pads_area/(4*device_parameter[CAD_manager.device_type]['ground_capacitance_pads_parameters']['width_of_pads'])

if pads_area <= 0.:
    SNAIL.ground_capacitance_pads_height = 0
    SNAIL.pads = False
else:
    SNAIL.ground_capacitance_pads_height = pads_height
SNAIL.ground_capacitance_pads_width = device_parameter[CAD_manager.device_type]['ground_capacitance_pads_parameters']['width_of_pads']

if device_parameter[CAD_manager.device_type]['TWPA_parameters']['inverted_cells'] == 'False':
    SNAIL.invert_SNAIL = False
else: 
    SNAIL.invert_SNAIL = False

In [91]:
# Layers definition
CAD_manager.pad_bottom_layer = pad_bottom_layer
CAD_manager.connection_wire_bottom_layer = connection_wire_bottom_layer
SNAIL.jj_bottom_layer = jj_bottom_layer
SNAIL.jj_top_layer = jj_top_layer
SNAIL.small_jj_top_layer = small_jj_top_layer
CAD_manager.pad_top_layer = pad_top_layer
CAD_manager.connection_wire_top_layer = connection_wire_top_layer
CAD_manager.ground_layer = ground_layer

In [92]:
# CAD manager device info
CAD_manager.n_unit_cells = device_parameter[CAD_manager.device_type]['TWPA_parameters']['number_of_cells']
CAD_manager.element = SNAIL
CAD_manager.dev_label = CAD_manager.device_type + '_V' + device_version + '_' + CAD_manager.chip
CAD_manager.dev_label += ' | ' + str(CAD_manager.n_unit_cells)
CAD_manager.design_filename = CAD_manager.device_type + '_' + CAD_manager.chip

In [93]:
# gds and job files generation
CAD_manager.generate_GDS()
CAD_manager.populate_dose_matric()
CAD_manager.generate_jobs(row=row, column=column)

### Modulation

In [94]:
TWPA_modulation = True

In [95]:
# Type of device
CAD_manager.device_type = 'SNAIL'
device_version = '01'

In [96]:
# Chip
row = 1
column = 1

In [97]:
### Fab parameters file
dirname = os.path.abspath('')
fab_param_filename = os.path.join(dirname, 'default_parameters\\fab_parameters.json')
with open(fab_param_filename) as json_file:
    fab_parameters = json.load(json_file)

In [98]:
### Device parameters file
dirname = os.path.abspath('')
device_param_filename = os.path.join(dirname, 'default_parameters\\device_parameters.json')
with open(device_param_filename) as json_file:
    device_parameter = json.load(json_file)

In [99]:
# CAD manager general info
CAD_manager.modulation = TWPA_modulation
CAD_manager.litho_plan = litho_plan
CAD_manager.dose_matric = dose_matric
CAD_manager.chip = str(row)+str(column)

In [100]:
# Device parameters
params = TWPA_parameters_cls.SNAIL_parameters(fab_parameters, device_parameter[CAD_manager.device_type])
print('\n'+'\033[1m'+'Chip '+CAD_manager.chip+'\033[0m')
Cg, total_ground_area, Hj_small, Wj_small, area_ratio = params.get_params(print_params=True, modulation = CAD_manager.modulation)


Chip 11
Junction capacitance per unit area: c_j = 45.00 fF/um^2
Large junction critical current density: j_c = 90.00 A/cm^2
Small junction critical current density: j_c = 105.00 A/cm^2
Number of cells: Ncell = 700
Number of large junctions in SNAIL: Njj = 3
Number of junctions per cell: Njj = 2
Large Junction dimensions: H = 4.00 um | W = 1.00 um | A = 4.00 um^2
Small Junction dimensions: H = 0.70 um | W = 0.49 um | A = 0.34 um^2
Critical current ratio: r_Ic = 0.10 
Area ratio: r_A = 0.09 
Critical current large: Ic = 2.77 uA
Critical current small: Ic = 0.28 uA
Josephson inductance per unit cell: Lj_cell @ 0.37 $\Phi_0$ = 416.88 pH
Josephson capacitance per unit cell: Cj_cell = 75.43 fF
Ground capacitance to get 50 Ohm matching: Cg = 166.75 fF
Total area to ground: 115.31 um^2
Plasma frequency: fj = 28.38 GHz
Cut-off frequency: f0 = 19.09 GHz
Modulation period: Np = 8
Modulation amplitude: eta = 10%
Gap frequency: fgap = 7.25 GHz


In [101]:
# Element design definition
SNAIL = TWPA_elements_cls.SNAILcell()
SNAIL.modulation = str(TWPA_modulation)
SNAIL.modulate_capa_only = device_parameter[CAD_manager.device_type]['TWPA_parameters']['modulate_capa_only']
SNAIL.keep_Z_constant_with_modulation = device_parameter[CAD_manager.device_type]['TWPA_parameters']['keep_Z_constant_with_modulation']
SNAIL.litho_overlap = 0.1
SNAIL.etching_offset = 0.25
SNAIL.area_ratio = area_ratio
SNAIL.width_jj_large = device_parameter[CAD_manager.device_type]['large_junction_parameters']['width_of_junction']
SNAIL.height_jj_large = device_parameter[CAD_manager.device_type]['large_junction_parameters']['height_of_junction']
SNAIL.spacing_jj_large = device_parameter[CAD_manager.device_type]['large_junction_parameters']['spacing_between_junctions']
SNAIL.electrode_height_difference_jj = 0.5
SNAIL.number_of_large_junctions = device_parameter[CAD_manager.device_type]['large_junction_parameters']['number_of_large_junctions_in_the_SNAIL']

SNAIL.width_jj_small = Wj_small
SNAIL.height_jj_small = Hj_small

SNAIL.loop_area = SNAIL.loop_area = device_parameter[CAD_manager.device_type]['loop_parameters']['loop_area']
SNAIL.loop_arm_width = SNAIL.spacing_jj_large/2
SNAIL.loop_width = SNAIL.number_of_large_junctions * (SNAIL.width_jj_large + SNAIL.spacing_jj_large) # Defined as is because the size of bottom loop arms is calculated upon the size of a JJ element (1*JJ_width + 2*JJ_spacing)
SNAIL.loop_height = SNAIL.loop_area / (SNAIL.loop_width - SNAIL.loop_arm_width)

upper_area = SNAIL.number_of_large_junctions * (SNAIL.width_jj_large + SNAIL.spacing_jj_large/2) * (2*SNAIL.electrode_height_difference_jj + SNAIL.height_jj_large) + SNAIL.number_of_large_junctions * SNAIL.height_jj_large * SNAIL.spacing_jj_large/2

loop_bridges_area = SNAIL.loop_arm_width*SNAIL.loop_height + SNAIL.loop_arm_width*(SNAIL.loop_height + SNAIL.height_jj_large + 2*SNAIL.electrode_height_difference_jj)

lower_area = (SNAIL.loop_width + SNAIL.loop_arm_width)*(SNAIL.height_jj_small + 2*SNAIL.electrode_height_difference_jj) - 2*SNAIL.electrode_height_difference_jj*(SNAIL.spacing_jj_large/2 - SNAIL.litho_overlap/2)


pads_area = total_ground_area - (upper_area + loop_bridges_area + lower_area)
pads_height = pads_area/(4*device_parameter[CAD_manager.device_type]['ground_capacitance_pads_parameters']['width_of_pads'])

if pads_area <= 0.:
    SNAIL.ground_capacitance_pads_height = 0
    SNAIL.pads = False
else:
    SNAIL.ground_capacitance_pads_height = pads_height
SNAIL.ground_capacitance_pads_width = device_parameter[CAD_manager.device_type]['ground_capacitance_pads_parameters']['width_of_pads']

if device_parameter[CAD_manager.device_type]['TWPA_parameters']['inverted_cells'] == 'False':
    SNAIL.invert_SNAIL = False
else: 
    SNAIL.invert_SNAIL = True
print('Pads area : %.2f um^2' %(round(pads_area,4)))

Pads area : 69.34 um^2


In [102]:
# Layer definition
CAD_manager.pad_bottom_layer = pad_bottom_layer
CAD_manager.connection_wire_bottom_layer = connection_wire_bottom_layer
SNAIL.jj_bottom_layer = jj_bottom_layer
SNAIL.jj_top_layer = jj_top_layer
SNAIL.small_jj_top_layer = small_jj_top_layer
CAD_manager.pad_top_layer = pad_top_layer
CAD_manager.connection_wire_top_layer = connection_wire_top_layer
CAD_manager.ground_layer = ground_layer

In [103]:
# CAD manager device info
CAD_manager.element = SNAIL
CAD_manager.n_unit_cells = device_parameter[CAD_manager.device_type]['TWPA_parameters']['number_of_cells']
CAD_manager.modulation_period = device_parameter[CAD_manager.device_type]['TWPA_parameters']['modulation_period']
CAD_manager.modulation_amplitude_percent = device_parameter[CAD_manager.device_type]['TWPA_parameters']['modulation_amplitude_percent']
CAD_manager.dev_label = CAD_manager.device_type + '_V' + device_version + '_' + CAD_manager.chip
CAD_manager.dev_label += ' | ' + str(CAD_manager.n_unit_cells) + ' | '+str(CAD_manager.modulation_period)+' | '+str(CAD_manager.modulation_amplitude_percent)+'%'
CAD_manager.design_filename = CAD_manager.device_type + '_' + CAD_manager.chip

In [104]:
# gds and job file generation
CAD_manager.generate_GDS()
CAD_manager.populate_dose_matric()
CAD_manager.generate_jobs(row=row, column=column)

## Resonator

### Lumped element

If a given device on the wafer has different number of layers and/or doses we can change the dose_matric variable

In [105]:
importlib.reload(TWPA_parameters_cls)
CAD_manager.load_reset_libs()

In [106]:
dose_matric = {str(ground_layer):3,
               str(capa_top_layer):14,
               str(pad_top_layer):15,
               str(connection_wire_top_layer):14,
              }

In [107]:
# Type of device
CAD_manager.device_type = 'LER'
device_version = '01'

In [108]:
# Chip
row = 1
column = 2

In [109]:
### Fab parameters file
dirname = os.path.abspath('')
fab_param_filename = os.path.join(dirname, 'default_parameters\\fab_parameters.json')
with open(fab_param_filename) as json_file:
    fab_parameters = json.load(json_file)

In [110]:
### Device parameters file
dirname = os.path.abspath('')
device_param_filename = os.path.join(dirname, 'default_parameters\\device_parameters.json')
with open(device_param_filename) as json_file:
    device_parameter = json.load(json_file)

In [111]:
# CAD manager general info
CAD_manager.modulation = False
CAD_manager.chip = str(row)+str(column)
CAD_manager.litho_plan = litho_plan
CAD_manager.dose_matric = dose_matric

In [112]:
# Device parameters
params = TWPA_parameters_cls.LER_parameters(fab_parameters, device_parameter[CAD_manager.device_type])
print('\n'+'\033[1m'+'Chip '+CAD_manager.chip+'\033[0m')
area_capa = params.get_params(print_params=True)
number_of_resonators = len(area_capa)


Chip 12
Capacitor capacitance per unit area: c = 8.63 fF/um^2
Number of capacitors: N_capa = 2
Meander length: l_meander = 2.93 mm
Coupling length: l_coupling = 300 um
fr (GHz) | C (fF) | A (um^2) | coupling (um)
11.21 	 | 40 	 | 9.27 	 | 89
10.20 	 | 55 	 | 12.75 	 | 87
9.66 	 | 65 	 | 15.06 	 | 85
8.99 	 | 80 	 | 18.54 	 | 83
8.27 	 | 100 	 | 23.17 	 | 81
7.70 	 | 120 	 | 27.81 	 | 79
7.13 	 | 145 	 | 33.60 	 | 77
6.43 	 | 185 	 | 42.87 	 | 75
5.90 	 | 225 	 | 52.14 	 | 73
5.19 	 | 300 	 | 69.52 	 | 71
4.54 	 | 400 	 | 92.70 	 | 69
4.09 	 | 500 	 | 115.87 	 | 67
3.75 	 | 600 	 | 139.05 	 | 65


In [113]:
# Element design definition
feedline = TWPA_elements_cls.FeedLine()
LER = TWPA_elements_cls.Overlap_LER()
feedline.feedline_length = CAD_manager.grid_size[0]*1000
feedline.feedline_width = device_parameter[CAD_manager.device_type]['feedline_parameters']['width_of_feedline'] 
feedline.feedline_gap = device_parameter[CAD_manager.device_type]['feedline_parameters']['gap_of_feedline'] 
feedline.x_pad = 150 
feedline.y_pad = 300 
feedline.x_arm = 20 
feedline.y_arm = feedline.feedline_width 
feedline.taper_length = 25 
feedline.x_pad_gap = 35 
feedline.y_pad_gap = 175
feedline.arm_gap = feedline.feedline_gap 
feedline.CPW = True
feedline.Exclude_ground = True
LER.resonators_ground_gap = 30
LER.width_of_wire = device_parameter[CAD_manager.device_type]['meander_parameters']['width_of_wire']
LER.spacing_between_steps = device_parameter[CAD_manager.device_type]['meander_parameters']['spacing_between_steps']
LER.length_of_step = device_parameter[CAD_manager.device_type]['meander_parameters']['length_of_step']
LER.length_of_coupling_step = device_parameter[CAD_manager.device_type]['meander_parameters']['length_of_coupling_step']
LER.length_of_meander = device_parameter[CAD_manager.device_type]['meander_parameters']['length_of_meander']
LER.number_of_capacitors = device_parameter[CAD_manager.device_type]['capacitor_parameters']['number_of_capacitors']
LER.spacing_capa = device_parameter[CAD_manager.device_type]['capacitor_parameters']['spacing_between_capa']

In [114]:
# Layer definition
LER.ground_layer = ground_layer
LER.capa_top_layer = capa_top_layer
CAD_manager.pad_top_layer = pad_top_layer
CAD_manager.connection_wire_top_layer = connection_wire_top_layer
CAD_manager.ground_layer = LER.ground_layer
CAD_manager.pad_bottom_layer = LER.bottom_layer
CAD_manager.connection_wire_bottom_layer = LER.bottom_layer
feedline.ground_layer = LER.ground_layer
feedline.feedline_layer = LER.bottom_layer

In [115]:
# CAD manager device info
CAD_manager.resonators_coupling_gap = np.asarray(device_parameter[CAD_manager.device_type]['resonator_parameters']['coupling_distance'])
CAD_manager.resonators = list([] for _ in range(number_of_resonators))
for j in range(0,number_of_resonators):
    LER.area_capa = area_capa[j]
    if j%2 == 0:
        CAD_manager.resonators[j] = LER.generateCell(reflection=True)
    else:
        CAD_manager.resonators[j] = LER.generateCell(reflection=False)
CAD_manager.feedline = feedline
CAD_manager.element = LER
CAD_manager.n_unit_cells = number_of_resonators
CAD_manager.dev_label = CAD_manager.device_type + '_V' + device_version + '_' + CAD_manager.chip
CAD_manager.design_filename = CAD_manager.device_type + '_' + CAD_manager.chip

In [116]:
# gds and job file generation
CAD_manager.generate_GDS()
CAD_manager.populate_dose_matric()
CAD_manager.generate_jobs(row=row, column=column)

# Job & Batch files generation

In [117]:
CAD_manager.batch_plan = batch_plan
CAD_manager.generate_batch_files()

# Wafer generation

Let's say we now want to write a wafer of 16 chips of the same kind of device varying some parameters from chip to chip.

In [3]:
date = '250506'
WaferName = 'SHGR'
WaferNumber = '02'

In [4]:
importlib.reload(CAD_manager_cls)
CAD_manager = CAD_manager_cls.CAD_manager(date, WaferName, WaferNumber)

Junk folder already exists, cleaning it...
Junk folder already exists, cleaning it...


In [5]:
importlib.reload(TWPA_parameters_cls)
CAD_manager.load_reset_libs()

## Layers

In [6]:
ground_layer = 1
pad_bottom_layer = 2
connection_wire_bottom_layer = 3
jj_bottom_layer = 4
small_jj_top_layer = 5
jj_top_layer = 6
pad_top_layer = 8
connection_wire_top_layer = 9

## Litho plan

In [7]:
litho_plan = {'ground':[ground_layer],
              'pads_bottom_layer':[pad_bottom_layer,connection_wire_bottom_layer],
              'bottom_layer':[jj_bottom_layer],
              'small_junction_top_layer':[small_jj_top_layer],
              'junctions_top_layer':[jj_top_layer],
              'pads_top_layer':[pad_top_layer,connection_wire_top_layer],
             }

In [8]:
batch_plan = {'ground':{'jobs':['ground'],                                                                                                               
                        'current':[15],
                        'datum':[8],
                        'sleep':[10]},
              'bottom_layer':{'jobs':['bottom_layer','pad_bottom_layer'],
                              'current':[3,30],
                              'datum':[8,8],
                              'sleep':[10,20]},
              'jj_top_layer':{'jobs':['small_junction_top_layer','junctions_top_layer'],
                              'current':[1,3],
                              'datum':[8,8],
                              'sleep':[10,10]},
             }

## Doses

In [9]:
dose_matric = {str(ground_layer):1,
               str(pad_bottom_layer):15,
               str(connection_wire_bottom_layer):15,
               str(jj_bottom_layer):16,
               str(jj_top_layer):16,
               str(pad_top_layer):15,
               str(connection_wire_top_layer):15,
               str(small_jj_top_layer):20,
              }

## Design

Let's add the RH TWPAs with modulation: both the modulation and the area of the junctions will vary.

In [10]:
CAD_manager.device_type = 'SNAIL'
device_version = '02'

In [11]:
rows = [0, 1, 2, 3]
columns = [0, 1, 2, 3]

In [ ]:
flux_for_50Ohm = [[0.37, 0.37, 0.37, 0.37], # 00 01 02 03
                  [0.39, 0.41, 0.41, 0.43], # 10 11 12 13
                  [0.39, 0.41, 0.41, 0.43], # 20 21 22 23
                  [0.43, 0.45, 0.45, 0.47]] # 30 31 32 33

Hj_large_mat = [[2, 2, 2, 2],
            [2, 2, 2, 2],
            [4, 4, 4, 4],
            [4, 4, 4, 4]]

Hj_small_mat = [[0.5, 0.5, 0.5, 0.5],
            [0.5, 0.5, 0.5, 0.5],
            [0.7, 0.7, 0.7, 0.7],
            [0.7, 0.7, 0.7, 0.7]]



r_ratio = [[0.13, 0.15, 0.15, 0.17],
           [0.18, 0.20, 0.20, 0.22],
           [0.18, 0.20, 0.20, 0.22],
           [0.23, 0.25, 0.25, 0.27]]

Np = [[8, 8, 8, 8],
     [10, 10, 10, 10],
     [10, 10, 10, 10],
     [8, 8, 8, 8]]

eta = [[15, 12, 15, 20],
       [0, 10, 12, 15],
       [0, 10, 12, 15],
       [12, 15, 15, 20]]

TWPA_modulation = [[True, True, True, True],
                   [False, True, True, True],
                   [False, True, True, True],
                   [True, True, True, True]]


In [13]:
### Fab parameters file
dirname = os.path.abspath('')
fab_param_filename = os.path.join(dirname, 'default_parameters\\fab_parameters.json')
with open(fab_param_filename) as json_file:
    fab_parameters = json.load(json_file)

### Device parameters file
dirname = os.path.abspath('')

device_param_filename = os.path.join(dirname, 'default_parameters\\device_parameters.json')
with open(device_param_filename) as json_file:
    device_parameter = json.load(json_file)

In [ ]:
for i in range(len(rows)):
    for j in range(len(columns)):
        CAD_manager.load_reset_libs()
        CAD_manager.modulation = TWPA_modulation[i]
        CAD_manager.litho_plan = litho_plan
        CAD_manager.dose_matric = dose_matric
        CAD_manager.chip = str(rows[i])+str(columns[j])


        device_parameter[CAD_manager.device_type]['large_junction_parameters']['height_of_junction'] = Hj_large_mat[i][j]
        device_parameter[CAD_manager.device_type]['small_junction_parameters']['height_of_junction'] = Hj_small_mat[i][j]
        device_parameter[CAD_manager.device_type]['TWPA_parameters']['modulation_period'] = Np[i][j]
        device_parameter[CAD_manager.device_type]['TWPA_parameters']['modulation_amplitude_percent'] = eta[i][j]
        device_parameter[CAD_manager.device_type]['small_junction_parameters']['critical_current_ratio_large_small'] = r_ratio[i][j]
        device_parameter[CAD_manager.device_type]['ground_capacitance_pads_parameters']['flux_impedance_matching'] = flux_for_50Ohm[i][j]
        
        params = TWPA_parameters_cls.SNAIL_parameters(fab_parameters, device_parameter[CAD_manager.device_type])
        print('\n'+'\033[1m'+'Chip '+CAD_manager.chip+'\033[0m')
        Cg, total_ground_area, Hj_small, Wj_small, area_ratio = params.get_params(print_params=True, modulation = CAD_manager.modulation)
        
        # Element design definition
        SNAIL = TWPA_elements_cls.SNAILcell()
        SNAIL.modulation = 'False'
        SNAIL.modulate_capa_only = device_parameter[CAD_manager.device_type]['TWPA_parameters']['modulate_capa_only']
        SNAIL.keep_Z_constant_with_modulation = device_parameter[CAD_manager.device_type]['TWPA_parameters']['keep_Z_constant_with_modulation']
        SNAIL.litho_overlap = 0.1
        SNAIL.etching_offset = 0.25
        SNAIL.area_ratio = area_ratio
        SNAIL.width_jj_large = device_parameter[CAD_manager.device_type]['large_junction_parameters']['width_of_junction']
        SNAIL.height_jj_large = device_parameter[CAD_manager.device_type]['large_junction_parameters']['height_of_junction']
        SNAIL.spacing_jj_large = device_parameter[CAD_manager.device_type]['large_junction_parameters']['spacing_between_junctions']
        SNAIL.electrode_height_difference_jj = 0.5
        SNAIL.number_of_large_junctions = device_parameter[CAD_manager.device_type]['large_junction_parameters']['number_of_large_junctions_in_the_SNAIL']

        SNAIL.width_jj_small = Wj_small
        SNAIL.height_jj_small = Hj_small

        SNAIL.loop_area = SNAIL.loop_area = device_parameter[CAD_manager.device_type]['loop_parameters']['loop_area']
        SNAIL.loop_arm_width = SNAIL.spacing_jj_large/2
        SNAIL.loop_width = SNAIL.number_of_large_junctions * (SNAIL.width_jj_large + SNAIL.spacing_jj_large) # Defined as is because the size of bottom loop arms is calculated upon the size of a JJ element (1*JJ_width + 2*JJ_spacing)
        SNAIL.loop_height = SNAIL.loop_area / (SNAIL.loop_width - SNAIL.loop_arm_width)

        upper_area = SNAIL.number_of_large_junctions * (SNAIL.width_jj_large + SNAIL.spacing_jj_large/2) * (2*SNAIL.electrode_height_difference_jj + SNAIL.height_jj_large) + SNAIL.number_of_large_junctions * SNAIL.height_jj_large * SNAIL.spacing_jj_large/2

        loop_bridges_area = SNAIL.loop_arm_width*SNAIL.loop_height + SNAIL.loop_arm_width*(SNAIL.loop_height + SNAIL.height_jj_large + 2*SNAIL.electrode_height_difference_jj)

        lower_area = (SNAIL.loop_width + SNAIL.loop_arm_width)*(SNAIL.height_jj_small + 2*SNAIL.electrode_height_difference_jj) - 2*SNAIL.electrode_height_difference_jj*(SNAIL.spacing_jj_large/2 - SNAIL.litho_overlap/2)


        pads_area = total_ground_area - (upper_area + loop_bridges_area + lower_area)
        pads_height = pads_area/(4*device_parameter[CAD_manager.device_type]['ground_capacitance_pads_parameters']['width_of_pads'])

        if pads_area <= 0.:
            SNAIL.ground_capacitance_pads_height = 0
            SNAIL.pads = False
        else:
            SNAIL.ground_capacitance_pads_height = pads_height
        SNAIL.ground_capacitance_pads_width = device_parameter[CAD_manager.device_type]['ground_capacitance_pads_parameters']['width_of_pads']

        if device_parameter[CAD_manager.device_type]['TWPA_parameters']['inverted_cells'] == 'False':
            SNAIL.invert_SNAIL = False
        else: 
            SNAIL.invert_SNAIL = True
        print('Pads area : %.2f um^2' %(round(pads_area,4)))
                
        CAD_manager.element = SNAIL
        CAD_manager.n_unit_cells = device_parameter[CAD_manager.device_type]['TWPA_parameters']['number_of_cells']
        CAD_manager.modulation_period = device_parameter[CAD_manager.device_type]['TWPA_parameters']['modulation_period']
        CAD_manager.modulation_amplitude_percent = device_parameter[CAD_manager.device_type]['TWPA_parameters']['modulation_amplitude_percent']
        CAD_manager.dev_label = CAD_manager.device_type + '_' + CAD_manager.chip
        CAD_manager.dev_label = WaferName + '_V' + WaferNumber + '_' + CAD_manager.chip
        CAD_manager.dev_label += ' | ' + str(CAD_manager.n_unit_cells) + ' | '+str(CAD_manager.modulation_period)+' | '+str(CAD_manager.modulation_amplitude_percent)+'%'
        CAD_manager.design_filename = WaferName + '_V' + WaferNumber + '_' + CAD_manager.chip

        CAD_manager.pad_bottom_layer = pad_bottom_layer
        CAD_manager.connection_wire_bottom_layer = connection_wire_bottom_layer
        SNAIL.jj_bottom_layer = jj_bottom_layer
        SNAIL.jj_top_layer = jj_top_layer
        SNAIL.small_jj_top_layer = small_jj_top_layer
        CAD_manager.pad_top_layer = pad_top_layer
        CAD_manager.connection_wire_top_layer = connection_wire_top_layer
        CAD_manager.ground_layer = ground_layer
        
        CAD_manager.generate_GDS()
        CAD_manager.populate_dose_matric()
        CAD_manager.generate_jobs(row=rows[i], column=columns[j])


Chip 00
Junction capacitance per unit area: c_j = 45.00 fF/um^2
Large junction critical current density: j_c = 90.00 A/cm^2
Small junction critical current density: j_c = 110.00 A/cm^2
Number of cells: Ncell = 700
Number of large junctions in SNAIL: Njj = 3
Number of junctions per cell: Njj = 2
Large Junction dimensions: H = 2.00 um | W = 1.00 um | A = 2.00 um^2
Small Junction dimensions: H = 0.50 um | W = 0.43 um | A = 0.21 um^2
Critical current ratio: r_Ic = 0.13 
Area ratio: r_A = 0.11 
Critical current large: Ic = 1.38 uA
Critical current small: Ic = 0.18 uA
Josephson inductance per unit cell: Lj_cell @ 0.37 $\Phi_0$ = 845.39 pH
Josephson capacitance per unit cell: Cj_cell = 39.57 fF
Ground capacitance to get 50 Ohm matching: Cg = 338.16 fF
Total area to ground: 194.85 um^2
Plasma frequency: fj = 27.52 GHz
Cut-off frequency: f0 = 9.41 GHz
No modulation
Pads area : 163.19 um^2

Chip 01
Junction capacitance per unit area: c_j = 45.00 fF/um^2
Large junction critical current density: 

## Job & Batch files generation

In [15]:
CAD_manager.batch_plan = batch_plan
CAD_manager.generate_batch_files()